# *Implementing SVC and RandomForest Classfier*
also performing Randomsearch and undersampling


In [72]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
import warnings
warnings.filterwarnings('ignore')
import pickle
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier


# *Importing and Cleaning the Data*

In [73]:
df=pd.read_csv('/content/data.csv')
df.head()
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [74]:
df.dropna(inplace=True)
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [75]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
df.dropna(inplace=True)
df.drop(columns=['customerID'],inplace=True)
df.head()
print(df['Churn'].value_counts())

Churn
No     5163
Yes    1869
Name: count, dtype: int64


In [76]:
df.describe()
y = df['Churn']
x = df.drop(columns=['Churn'], axis=1)
print(x.shape)
print(y.shape)
x = pd.get_dummies(x, drop_first=True)
x
y=pd.get_dummies(y,drop_first=True)
y

(7032, 19)
(7032,)


,Yes
0,False
1,False
2,True
3,False
4,True
...,...
7038,False
7039,False
7040,False
7041,True


In [77]:
from imblearn.under_sampling import RandomUnderSampler

# Initialize the sampler (random_state ensures reproducibility)
rus = RandomUnderSampler(random_state=42, sampling_strategy='majority')

# Fit and resample the data
X_resampled, y_resampled = rus.fit_resample(x, y)
print(y_resampled['Yes'].value_counts())

Yes
False    1869
True     1869
Name: count, dtype: int64


In [78]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_sc=sc.fit_transform(X_resampled)

In [79]:
x_train,x_test,y_train,y_test=train_test_split(x_sc,y_resampled,test_size=0.2,random_state=42)
print(len(x_train))
print(len(x_test))

2990
748


In [80]:
rf=RandomForestClassifier()
rf.fit(x_train,y_train)
y_pred=rf.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

       False       0.77      0.74      0.75       379
        True       0.74      0.78      0.76       369

    accuracy                           0.76       748
   macro avg       0.76      0.76      0.76       748
weighted avg       0.76      0.76      0.76       748



# Implementing RandomSearchCV and finding the best hyperparameters

In [81]:
from sklearn.model_selection import RandomizedSearchCV
n_estimators=[int(x) for x in np.linspace(start=100,stop=100,num=10)]
max_depth=[int(x) for x in np.linspace(5,30,num=6)]
min_samples_split=[2,5,10,15,100]
min_samples_leaf=[1,2,5,10]
bootstrap=[True,False]
random_search={'n_estimators':n_estimators,
               'max_depth':max_depth,
               'min_samples_split':min_samples_split,
               'min_samples_leaf':min_samples_leaf,
               'bootstrap':bootstrap}
rf=RandomForestClassifier()
rf_random=RandomizedSearchCV(estimator=rf,param_distributions=random_search,n_iter=100,cv=3,verbose=2,random_state=42,n_jobs=-1)
rf_random.fit(x_train,y_train)
y_pred=rf_random.predict(x_test)

Fitting 3 folds for each of 100 candidates, totalling 300 fits


KeyboardInterrupt: 

In [82]:
best_random_grid=rf_random.best_params_
print(best_random_grid)

AttributeError: 'RandomizedSearchCV' object has no attribute 'best_params_'

Got best parameters for randomforest as {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 25, 'bootstrap': True}

In [83]:
rf=RandomForestClassifier(n_estimators= 100, min_samples_split= 2, min_samples_leaf= 2, max_depth= 25, bootstrap= True)
rf.fit(x_train,y_train)
y_pred=rf.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

       False       0.79      0.73      0.76       379
        True       0.74      0.80      0.77       369

    accuracy                           0.77       748
   macro avg       0.77      0.77      0.77       748
weighted avg       0.77      0.77      0.77       748



In [85]:
from sklearn.svm import SVC
svc=SVC()
svc.fit(x_train,y_train)
y_pred=svc.predict(x_test)
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

       False       0.72      0.78      0.75       349
        True       0.79      0.73      0.76       399

    accuracy                           0.75       748
   macro avg       0.75      0.76      0.75       748
weighted avg       0.76      0.75      0.75       748

